# Predicting whether a loan will default (Charged Off) vs be full repaid, using loan application and performance data from LendingClub 

Lenders lose money when borrowers fail to repay, so estimating the probability of default (PD) before approving a loan is a core part of credit risk management. This project uses historical LendingClub loan data (2007–2019) to predict whether a loan will be Fully Paid or Charged Off, based on information available when the loan was issued, such as loan amount, interest rate, income, debt-to-income ratio and credit history. Because defaults are much rarer than repayments, the project addresses class imbalance and evaluates models with precision, recall and ROC-AUC instead of accuracy. It also considers the different costs of approving a loan that defaults versus rejecting one that would have been repaid, to choose a decision threshold a lender could use in practice.

## Intial Setup & inspect structure
Target: default = 1 for Charged Off and Default, Fully Paid = 0. Excluded loans with no final outcome yet (Current, Late, In Grace Period) and early loans issued under a superseded credit policy. Result: 1,385,902 loans, about 19.8% defaulted.

In [13]:
import gc 
import pandas as pd

df_raw = pd.read_csv("../data/raw/appl_accepted_20072019Q3.csv", low_memory=False)
print(df_raw['loan_status'].value_counts())

df = df_raw[df_raw['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()

del df_raw
gc.collect()

print(df.shape)
df.to_parquet("../data/processed/loans_resolved.parquet", index=False)

loan_status
Current                                                1217855
Fully Paid                                             1111884
Charged Off                                             273973
Late (31-120 days)                                       27067
In Grace Period                                          11386
Late (16-30 days)                                         5558
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     45
Name: count, dtype: int64
(1385902, 150)


In [14]:
df = pd.read_parquet("../data/processed/loans_resolved.parquet")
df.info(verbose=True, show_counts=True)

# 0 = Fully Paid, 1 = Charged Off or Default
df['default'] = df['loan_status'].isin(['Charged Off', 'Default']).astype(int)
print(df['default'].value_counts(normalize=True))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1385902 entries, 0 to 1385901
Data columns (total 150 columns):
 #    Column                                      Non-Null Count    Dtype  
---   ------                                      --------------    -----  
 0    id                                          1385902 non-null  object 
 1    member_id                                   0 non-null        float64
 2    loan_amnt                                   1385902 non-null  float64
 3    funded_amnt                                 1385902 non-null  float64
 4    funded_amnt_inv                             1385902 non-null  float64
 5    term                                        1385902 non-null  object 
 6    int_rate                                    1385902 non-null  object 
 7    installment                                 1385902 non-null  float64
 8    grade                                       1385902 non-null  object 
 9    sub_grade                                   

### Missingness check

In [15]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
print(missing[missing > 0].to_string())

member_id                                     100.000000
next_pymnt_d                                   99.996753
orig_projected_additional_accrued_interest     99.728624
hardship_payoff_balance_amount                 99.584675
hardship_last_payment_amount                   99.584675
hardship_reason                                99.584675
hardship_type                                  99.584675
hardship_status                                99.584675
hardship_amount                                99.584675
deferral_term                                  99.584675
hardship_end_date                              99.584675
hardship_dpd                                   99.584675
hardship_loan_status                           99.584675
hardship_length                                99.584675
payment_plan_start_date                        99.584675
hardship_start_date                            99.584675
sec_app_mths_since_last_major_derog            99.412946
sec_app_revol_util             

In [16]:
# issue_d has two formats: 'Dec-2018' (most rows) and 'Mar-19' (2019 loans)
d = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df['issue_d'] = d.fillna(pd.to_datetime(df['issue_d'], format='%b-%y', errors='coerce'))
df['issue_year'] = df['issue_d'].dt.year
print('Unparsed dates:', df['issue_d'].isna().sum())

print(df.groupby('issue_year')['open_acc_6m'].apply(lambda s: s.isnull().mean()).round(2))

Unparsed dates: 0
issue_year
2007    1.00
2008    1.00
2009    1.00
2010    1.00
2011    1.00
2012    1.00
2013    1.00
2014    1.00
2015    0.95
2016    0.00
2017    0.00
2018    0.00
2019    0.00
Name: open_acc_6m, dtype: float64


### Drop columns

In [17]:
# Group 2: "has ever had X" flags, only where blank really means "never"
flag_cols = ['mths_since_last_delinq', 'mths_since_last_record']
for c in flag_cols:
    df['has_' + c.replace('mths_since_', '')] = df[c].notnull().astype(int)

# Not collected before 2012, so blank there doesn't mean "never". Drop for v1
uncollected = ['mths_since_last_major_derog', 'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq']

# Group 3: block only collected from late 2015
block_58 = ['open_acc_6m', 'inq_last_12m', 'total_cu_tl', 'open_il_12m', 'open_il_24m', 'open_rv_12m',
            'open_rv_24m', 'max_bal_bc', 'total_bal_il', 'open_act_il', 'inq_fi', 'all_util', 'il_util',
            'mths_since_rcnt_il']

# Group 1: empty, identifiers/free text, joint-only, hardship/settlement
drop_cols = (['member_id', 'id', 'url', 'desc', 'emp_title', 'title', 'zip_code', 'next_pymnt_d',
              'orig_projected_additional_accrued_interest', 'deferral_term', 'payment_plan_start_date',
              'revol_bal_joint', 'verification_status_joint', 'dti_joint', 'annual_inc_joint',
              'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_date']
             + [c for c in df.columns if c.startswith(('hardship_', 'settlement_', 'sec_app_'))])

# Single value across every row
constant = ['policy_code', 'pymnt_plan', 'hardship_flag']

# Post-outcome leakage: describes what happened after approval
leakage = ['out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
           'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
           'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low']

to_drop = list(dict.fromkeys(flag_cols + uncollected + block_58 + drop_cols + constant + leakage))
print('Names not found:', [c for c in to_drop if c not in df.columns])

df = df.drop(columns=to_drop, errors='ignore')
print(df.shape)

Names not found: []
(1385902, 73)


In [18]:
# Stored as text; convert so median imputation works later
for c in ['int_rate', 'revol_util']:
    df[c] = pd.to_numeric(df[c].astype(str).str.rstrip('%'), errors='coerce')

# Any remaining columns with a single value?
print('Constant columns:', [c for c in df.columns if df[c].nunique(dropna=False) <= 1])

print(df.columns.tolist())

Constant columns: []
['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',

### Missingness decisions
- **Flags:** `mths_since_last_delinq` and `mths_since_last_record` became 0/1 `has_*` flags, since a blank means the borrower never had that event.
- **Not-yet-collected columns:** `mths_since_last_major_derog`, `mths_since_recent_bc_dlq` and `mths_since_recent_revol_delinq` are 100% blank for 2007–2011 loans because LendingClub hadn't started collecting them, so a blank doesn't mean "never". Dropped for v1.
- **58% block:** 14 account-activity columns (`open_acc_6m`, `il_util`, etc.) are blank for every loan before late 2015. Dropped rather than imputed.
- **Leakage:** payment, recovery and settlement columns (`total_pymnt`, `recoveries`, `debt_settlement_flag`, etc.) describe what happened after approval, so they were removed.
- **Small gaps:** about 4.9% of loans (mostly pre-2012) are missing the `num_*`, `tot_*` and `mo_sin_*` fields. These will be imputed with the median inside the pipeline, after the train/test split, with missing-value indicators.

### Duplicate check

In [ ]:
# Duplicate loans: rows identical on every column (id was dropped above; it had no repeats either)
print('Duplicate rows:', df.duplicated().sum())

### Categorical variables

In [ ]:
# funded_amnt_inv is settled after listing (mild leakage); funded_amnt nearly duplicates loan_amnt
df = df.drop(columns=['funded_amnt', 'funded_amnt_inv'])

# term: ' 36 months' -> 36
df['term'] = df['term'].str.extract(r'(\d+)', expand=False).astype(int)

# emp_length: '< 1 year' -> 0, '10+ years' -> 10, blanks stay missing (imputed in the pipeline)
df['emp_length'] = (df['emp_length'].replace('< 1 year', '0 years')
                    .str.extract(r'(\d+)', expand=False).astype(float))

# sub_grade: A1 -> 1 ... G5 -> 35 (ordinal, finer than grade)
# grade is kept as letters for EDA and dropped before modelling
df['sub_grade_num'] = (df['sub_grade'].str[0].map({g: i for i, g in enumerate('ABCDEFG')}) * 5
                       + df['sub_grade'].str[1].astype(int))
df = df.drop(columns='sub_grade')

# home_ownership: merge the tiny ANY / NONE categories into OTHER
df['home_ownership'] = df['home_ownership'].replace({'ANY': 'OTHER', 'NONE': 'OTHER'})

# earliest_cr_line -> credit history length in months at issue date
# Same two formats as issue_d; '%b-%y' reads 'Mar-65' as 2065, so pull any future date back 100 years
ecl = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
ecl = ecl.fillna(pd.to_datetime(df['earliest_cr_line'], format='%b-%y', errors='coerce'))
ecl = ecl.where(ecl <= df['issue_d'], ecl - pd.DateOffset(years=100))
df['credit_history_months'] = ((df['issue_d'].dt.year - ecl.dt.year) * 12
                               + (df['issue_d'].dt.month - ecl.dt.month))
df = df.drop(columns='earliest_cr_line')

print('Unparsed earliest_cr_line:', ecl.isna().sum())
print(df[['term', 'emp_length', 'sub_grade_num', 'credit_history_months']].describe().round(1))
print(df.select_dtypes('object').nunique())

### Categorical encoding plan
- **Converted to numbers here:** `term` (36/60), `emp_length` (0–10, blanks imputed later), `sub_grade` → `sub_grade_num` (1–35, ordinal) and `earliest_cr_line` → `credit_history_months`.
- **One-hot in the pipeline (Phase 3):** `home_ownership` (ANY/NONE merged into OTHER), `verification_status`, `purpose`, `initial_list_status`, `application_type` and `addr_state`.
- **Kept for EDA, dropped before modelling:** `grade` (overlaps with `sub_grade_num`), `loan_status`, `issue_d`, `issue_year`.
- **Also dropped:** `funded_amnt_inv` (settled after listing) and `funded_amnt` (near-duplicate of `loan_amnt`).
- **Duplicates:** none found.

### Save cleaned dataset

In [ ]:
df.to_parquet("../data/processed/loans_clean.parquet", index=False)
print(df.shape)